## Runtime requirement

This notebook assumes a Google Colab runtime with GPU.

Before connecting, go to:
Runtime → Change runtime type → T4 GPU (for example)

In [ ]:
# @title GPU availability check
import torch
from IPython.display import display, HTML

# Show a visible warning message if GPU is not available
if not torch.cuda.is_available():
    display(HTML("""
    <div style="padding:12px; border-radius:8px; background-color:#fff3cd; color:#856404; border:1px solid #ffeeba;">
        <b>Warning:</b> GPU is not enabled.<br>
        Please go to <b>Runtime -> Change runtime type -> T4 GPU</b>.
    </div>
    """))
else:
    print("GPU is available:", torch.cuda.get_device_name(0))

## Installation requirements

In [ ]:
# requirements
!pip install freesasa biopython fair-esm joblib -q
!pip install scikit-learn==1.8.0 -q

## Importing modules

In [ ]:
# Clone the GitHub repository into Colab
!git clone https://github.com/fukasawa-group/prsurflm
%cd /content/prsurflm

In [ ]:
import os, sys
import argparse
sys.path.append("/content/prsurflm/")

import json
import numpy as np
import torch

print("imports OK")

In [ ]:
# @title ESM-2 preloading
# ESM-2 loading
import esm, torch

model_name = "esm2_t33_650M_UR50D"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fn = getattr(esm.pretrained, model_name)
_model, _alphabet = fn()          # 1.5 GB
_model = _model.to(device).eval()
_batch_converter = _alphabet.get_batch_converter()

print(f"ESM-2 loaded on {device}  |  layers={_model.num_layers}")
del _model, _alphabet, _batch_converter

In [ ]:
# @title PDB input
import os
import glob
import ipywidgets as widgets
from IPython.display import display, clear_output

# --------------------------------------------------
# Directories
# --------------------------------------------------
REPO_DIR = "/content/prsurflm"

# Bundled test PDB directory in the repository
TEST_ROOT_DIR = os.path.join(REPO_DIR, "test")
TEST_PDB_DIR = os.path.join(TEST_ROOT_DIR, "pdb")

# User-uploaded PDB directory in Colab
UPLOAD_PDB_DIR = "/content/pdb"

os.makedirs(UPLOAD_PDB_DIR, exist_ok=True)

# --------------------------------------------------
# Shared state for downstream inference
# --------------------------------------------------
input_state = {
    "mode": "demo",
    "base_name": None,
    "selected_pdb_path": None,
}

# --------------------------------------------------
# Helper functions
# --------------------------------------------------
def find_test_bases(test_pdb_dir):
    """Find valid bundled test examples based on xxx.pdb"""
    pdb_files = glob.glob(os.path.join(test_pdb_dir, "*.pdb"))
    return sorted([
        os.path.splitext(os.path.basename(p))[0]
        for p in pdb_files
    ])


def set_demo_path(base):
    """Set shared path from bundled test data"""
    input_state["mode"] = "demo"
    input_state["base_name"] = base
    input_state["selected_pdb_path"] = os.path.join(TEST_PDB_DIR, f"{base}.pdb")


def clear_selected_path():
    """Clear selected input path"""
    input_state["base_name"] = None
    input_state["selected_pdb_path"] = None


def input_is_ready():
    """Check whether the PDB path is available"""
    return input_state["selected_pdb_path"] is not None


# --------------------------------------------------
# Build demo mode UI
# --------------------------------------------------
test_bases = find_test_bases(TEST_PDB_DIR)

demo_title = widgets.HTML("<b>Bundled test PDB</b>")
demo_desc = widgets.HTML(
    "<p>Select one of the bundled PDB examples included in the repository.</p>"
)

if test_bases:
    demo_selector = widgets.Dropdown(
        options=test_bases,
        value=test_bases[0],
        description="Example:",
        layout=widgets.Layout(width="450px")
    )
else:
    demo_selector = widgets.Dropdown(
        options=[],
        value=None,
        description="Example:",
        layout=widgets.Layout(width="450px")
    )

demo_status = widgets.Output()


def refresh_demo_status():
    """Refresh the demo mode status panel"""
    with demo_status:
        clear_output()

        if not test_bases:
            print("No bundled test PDB files were found.")
            print("Expected directory:")
            print(" -", TEST_PDB_DIR)
            print("")
            print("Expected file pattern:")
            print(" - test/pdb/xxxx.pdb")
            clear_selected_path()
            return

        base = demo_selector.value
        if base is None:
            print("No bundled test example is selected.")
            clear_selected_path()
            return

        set_demo_path(base)

        print("✅ Bundled test PDB selected")
        print("Base name:", base)
        print("PDB:", input_state["selected_pdb_path"])


def on_demo_change(change):
    """Handle changes in the demo selector"""
    if change["name"] == "value":
        refresh_demo_status()


demo_selector.observe(on_demo_change, names="value")

demo_ui = widgets.VBox([
    demo_title,
    demo_desc,
    demo_selector,
    demo_status,
])

# --------------------------------------------------
# Build upload mode UI
# --------------------------------------------------
upload_title = widgets.HTML("<b>Upload your own PDB</b>")
upload_desc = widgets.HTML(
    "<p>Required file:</p>"
    "<ul>"
    "<li><code>xxxx.pdb</code></li>"
    "</ul>"
    "<p>The uploaded file will be saved under <code>/content/pdb</code>.</p>"
)

main_upload = widgets.FileUpload(
    accept=".pdb",
    multiple=False,
    description="PDB file"
)

upload_status = widgets.Output()


def handle_main_upload(change):
    """Handle upload of the main PDB file"""
    with upload_status:
        clear_output()

        if not main_upload.value:
            print("No PDB file selected.")
            return

        item = list(main_upload.value.values())[0]
        filename = item["metadata"]["name"]
        content = item["content"]

        if not filename.lower().endswith(".pdb"):
            print("Error: Please upload a .pdb file.")
            return

        save_path = os.path.join(UPLOAD_PDB_DIR, filename)
        with open(save_path, "wb") as f:
            f.write(content)

        base = os.path.splitext(filename)[0]

        input_state["mode"] = "upload"
        input_state["base_name"] = base
        input_state["selected_pdb_path"] = save_path

        print("✅ PDB uploaded successfully")
        print("File:", filename)
        print("Saved to:", save_path)


main_upload.observe(handle_main_upload, names="value")

upload_ui = widgets.VBox([
    upload_title,
    upload_desc,
    widgets.HTML("<b>Step 1: Upload PDB</b>"),
    main_upload,
    upload_status,
])

# --------------------------------------------------
# Top-level mode selector
# --------------------------------------------------
title = widgets.HTML("<h3>PDB Input</h3>")
desc = widgets.HTML(
    "<p>Choose one of the following options:</p>"
    "<ul>"
    "<li><b>Use bundled test PDB</b> (recommended for first-time users)</li>"
    "<li><b>Upload your own PDB</b></li>"
    "</ul>"
)

mode_selector = widgets.RadioButtons(
    options=[
        ("Use bundled test PDB", "demo"),
        ("Upload your own PDB", "upload"),
    ],
    value="demo",
    description="Mode:",
    layout=widgets.Layout(width="500px")
)

mode_status = widgets.Output()
mode_ui_box = widgets.VBox([])


def render_mode_ui(mode):
    """Render UI for the selected input mode"""
    if mode == "demo":
        mode_ui_box.children = [demo_ui]
        refresh_demo_status()
    else:
        mode_ui_box.children = [upload_ui]
        clear_selected_path()
        input_state["mode"] = "upload"
        with upload_status:
            clear_output()
            print("Upload mode selected.")
            print("Please upload:")
            print(" - xxxx.pdb")


def on_mode_change(change):
    """Handle switching between demo mode and upload mode"""
    if change["name"] != "value":
        return

    with mode_status:
        clear_output()
        if change["new"] == "demo":
            print("Demo mode selected.")
        else:
            print("Upload mode selected.")

    render_mode_ui(change["new"])


mode_selector.observe(on_mode_change, names="value")

# --------------------------------------------------
# Summary panel
# --------------------------------------------------
summary_title = widgets.HTML("<b>Current selection</b>")
summary_button = widgets.Button(description="Show selected input")
summary_output = widgets.Output()


def show_summary(_):
    """Show the currently selected input path"""
    with summary_output:
        clear_output()
        print("Mode:", input_state["mode"])
        print("Base name:", input_state["base_name"])
        print("PDB path:", input_state["selected_pdb_path"])
        print("Ready:", input_is_ready())


summary_button.on_click(show_summary)

# --------------------------------------------------
# Display the full UI
# --------------------------------------------------
ui = widgets.VBox([
    title,
    desc,
    mode_selector,
    mode_status,
    mode_ui_box,
    widgets.HTML("<hr>"),
    summary_title,
    summary_button,
    summary_output,
])

display(ui)

# Initialize the default view
render_mode_ui(mode_selector.value)

## Configuration

In [ ]:
if not input_is_ready():
    raise ValueError("Input files are not fully set. Please select bundled test data or upload all required files.")

# Selected input from the widget
PDB_PATH     = input_state["selected_pdb_path"]

# Derived path components
PDB_DIR        = os.path.dirname(PDB_PATH)
PDB            = os.path.basename(PDB_PATH)

# Fixed settings
RECEPTOR   = "A"
LIGAND     = "B"
CHECKPOINT = "/content/prsurflm/checkpoint/lr_nr_model.joblib"
DEVICE     = "cuda:0"

print("PDB_PATH:", PDB_PATH)
print("CHECKPOINT:", CHECKPOINT)
print("DEVICE:", DEVICE)

In [ ]:
!python predict_ppi_esm_lr.py \
    --pdb {PDB_DIR}/{PDB} \
    --receptor       {RECEPTOR} \
    --ligand         {LIGAND} \
    --lr_model       {CHECKPOINT} \
    --device         {DEVICE} \
    --save_json result.json